In [ ]:
# ─── Cell 1: Installs ────────────────────────────────────────
%pip install alpaca-py
%pip install pandas numpy plotly vectorbt


Note: you may need to restart the kernel to use updated packages.


In [2]:
# ─── Cell 2: Import Libraries ────────────────────────────────
import pandas as pd
import numpy as np
import vectorbt as vbt
import plotly.graph_objects as go
from datetime import datetime, timedelta
from alpaca.data.historical import StockHistoricalDataClient
from alpaca.data.requests import StockBarsRequest
from alpaca.data.timeframe import TimeFrame, TimeFrameUnit

print("✅ All libraries imported successfully")
print(f"   pandas:    {pd.__version__}")
print(f"   numpy:     {np.__version__}")
print(f"   vectorbt:  {vbt.__version__}")
print(f"   alpaca-py: 0.43.5")

✅ All libraries imported successfully
   pandas:    2.2.2
   numpy:     1.26.4
   vectorbt:  1.0.0
   alpaca-py: 0.43.5


In [3]:
# ─── Cell 3: Alpaca Connection ────────────────────────────────
from dotenv import load_dotenv
import os

dotenv_path = r"c:\Users\kwgit\Desktop\Playbook\Claude_2025_2026_Playbooks\AlgoTrading\AlgoTrading\.env"
load_dotenv(dotenv_path, override=True)

API_KEY    = os.getenv("ALPACA_API_KEY")
SECRET_KEY = os.getenv("ALPACA_SECRET_KEY")

client = StockHistoricalDataClient(API_KEY, SECRET_KEY)

print("✅ Alpaca connected successfully")
print(f"   API Key loaded: {'✅' if API_KEY else '❌'}")
print(f"   Secret loaded:  {'✅' if SECRET_KEY else '❌'}")

✅ Alpaca connected successfully
   API Key loaded: ✅
   Secret loaded:  ✅


In [ ]:
# ─── Cell 4: Pull NVDL 5-MIN Data ────────────────────────────
from alpaca.data.requests import StockBarsRequest
from alpaca.data.timeframe import TimeFrame, TimeFrameUnit
from datetime import datetime

request_params = StockBarsRequest(
    symbol_or_symbols = ["NVDL"],
    timeframe          = TimeFrame(5, TimeFrameUnit.Minute),
    start              = datetime(2023, 1, 1),
    end                = datetime(2025, 8, 8),
)

bars    = client.get_stock_bars(request_params)
df_nvdl = bars.df.reset_index()

print(f"✅ Data loaded successfully")
print(f"   Symbol:  NVDL")
print(f"   Bars:    {len(df_nvdl):,}")
print(f"   From:    {df_nvdl.index.min()}")
print(f"   To:      {df_nvdl.index.max()}")
print(f"\nColumns: {df_nvdl.columns.tolist()}")

✅ Data loaded successfully
   Symbol:  NVDL
   Bars:    139,486
   From:    0
   To:      139485

Columns: ['symbol', 'timestamp', 'open', 'high', 'low', 'close', 'volume', 'trade_count', 'vwap']


In [5]:
# ─── Cell 5: Clean Data ───────────────────────────────────────
import pandas as pd

# In this notebook timestamp is a column after reset_index()
# Need to set it as index properly

df_nvdl['timestamp'] = pd.to_datetime(df_nvdl['timestamp'])

# Check if timezone aware
if df_nvdl['timestamp'].dt.tz is None:
    df_nvdl['timestamp'] = df_nvdl['timestamp'].dt.tz_localize('UTC')

# Convert to Eastern
df_nvdl['timestamp'] = df_nvdl['timestamp'].dt.tz_convert('America/New_York')

# Set as index
df_nvdl = df_nvdl.set_index('timestamp')

# Now copy and filter
df = df_nvdl.copy()
df = df.between_time('09:30', '16:00')
df = df.dropna()

print(f"✅ Data cleaned successfully")
print(f"   Total bars:  {len(df):,}")
print(f"   From:        {df.index.min()}")
print(f"   To:          {df.index.max()}")
print(f"   Columns:     {df.columns.tolist()}")
print(f"\nSample:")
print(df[['open','high','low','close','volume','vwap']].tail(5))

✅ Data cleaned successfully
   Total bars:  66,701
   From:        2023-01-03 09:45:00-05:00
   To:          2026-07-30 16:00:00-04:00
   Columns:     ['symbol', 'open', 'high', 'low', 'close', 'volume', 'trade_count', 'vwap']

Sample:
                             open     high      low    close    volume  \
timestamp                                                                
2026-07-30 15:40:00-04:00  27.620  27.7550  27.6100  27.7450  133525.0   
2026-07-30 15:45:00-04:00  27.745  27.8150  27.7450  27.7800   84971.0   
2026-07-30 15:50:00-04:00  27.790  27.9000  27.6864  27.8450  406151.0   
2026-07-30 15:55:00-04:00  27.840  27.8900  27.7350  27.8400  342988.0   
2026-07-30 16:00:00-04:00  27.840  28.0481  27.7500  28.0081   62016.0   

                                vwap  
timestamp                             
2026-07-30 15:40:00-04:00  27.715062  
2026-07-30 15:45:00-04:00  27.781072  
2026-07-30 15:50:00-04:00  27.790718  
2026-07-30 15:55:00-04:00  27.816798  
2026-07-30 

In [ ]:
# ─── Cell 6: Signals — LOCKED STRATEGY (No Changes Allowed) ──
# Strategy: 9-EMA + Prev Day High + Positive Open + VWAP + QQQ
# Morning Only (9:30A-12P)
# This is the EXACT strategy from 2023-2025 in-sample period
# DO NOT modify any parameters — out-of-sample test integrity

import numpy as np
import datetime
import pandas as pd

# ── 9-EMA ─────────────────────────────────────────────────────
df['ema9']      = df['close'].ewm(span=9, adjust=False).mean()
df['ema_slope'] = df['ema9'] - df['ema9'].shift(6)

# ── Core EMA filters ──────────────────────────────────────────
df['slope_filter']     = df['ema_slope'] > 0.15      # LOCKED
df['prev_touched_ema'] = df['low'].shift(1) <= df['ema9'].shift(1)
df['closed_above_ema'] = df['close'] > df['ema9']

# ── VWAP filter ───────────────────────────────────────────────
df['above_vwap'] = df['close'] > df['vwap']

# ── Stop + Risk ───────────────────────────────────────────────
df['stop'] = df['low'].shift(1)
df['risk']  = (df['close'] - df['stop']).clip(lower=0)

# ── Time ──────────────────────────────────────────────────────
df['time'] = df.index.time
df['date'] = df.index.date

# ── Previous Day High filter ──────────────────────────────────
daily_high = df.groupby('date')['high'].max()
df['prev_day_high'] = df['date'].map(
    lambda d: daily_high.get(
        (pd.Timestamp(d) - pd.offsets.BDay(1)).date(),
        np.nan
    )
)
df['above_prev_high'] = df['close'] > df['prev_day_high']

# ── Positive Open filter ──────────────────────────────────────
daily_close = df.groupby('date')['close'].last()
df['prev_day_close'] = df['date'].map(
    lambda d: daily_close.get(
        (pd.Timestamp(d) - pd.offsets.BDay(1)).date(),
        np.nan
    )
)
df['day_open']      = df.groupby('date')['open'].transform('first')
df['positive_open'] = df['day_open'] > df['prev_day_close']

# ── QQQ filter ────────────────────────────────────────────────
df_qqq['ema9']       = df_qqq['close'].ewm(span=9, adjust=False).mean()
df_qqq['above_vwap'] = df_qqq['close'] > df_qqq['vwap']
df_qqq['above_ema']  = df_qqq['close'] > df_qqq['ema9']

df['qqq_above_vwap'] = df_qqq['above_vwap']
df['qqq_above_ema']  = df_qqq['above_ema']
df['qqq_filter']     = (df['qqq_above_vwap'] & df['qqq_above_ema']).fillna(False)

# ── Morning session ───────────────────────────────────────────
morning = (
    (df['time'] >= datetime.time(9, 30)) &
    (df['time'] < datetime.time(12, 0))
)

# ── LOCKED Entry Signal ───────────────────────────────────────
df['signal'] = (
    df['prev_touched_ema']  &
    df['closed_above_ema']  &
    df['slope_filter']      &
    df['above_vwap']        &
    df['above_prev_high']   &
    df['positive_open']     &
    df['qqq_filter']        &
    morning
).astype(int)

print(f"✅ LOCKED Strategy Signals — 2026 Out-of-Sample")
print(f"   Filters: 9-EMA + Prev High + Pos Open + VWAP + QQQ + Morning")
print(f"   Total signals: {df['signal'].sum():,}")
print(f"\nFilter breakdown:")
print(f"   slope_filter:     {df['slope_filter'].sum():,}")
print(f"   above_vwap:       {df['above_vwap'].sum():,}")
print(f"   above_prev_high:  {df['above_prev_high'].sum():,}")
print(f"   positive_open:    {df['positive_open'].sum():,}")
print(f"   qqq_filter:       {df['qqq_filter'].sum():,}")
print(f"   morning session:  {morning.sum():,}")

EMA Slope statistics:
   Mean:    $0.0008
   Median:  $0.0238
   75th:    $0.2840
   90th:    $0.6602

✅ Signals calculated — EMA Slope Filter ONLY
   Total signals:     5,278
   Slope filter bars: 23,981

Sample signals:
                             close       ema9   stop    risk
timestamp                                                   
2023-01-24 12:30:00-05:00  27.8800  27.832177  27.82  0.0600
2023-01-27 15:40:00-05:00  30.8305  30.745209  30.72  0.1105
2023-02-01 09:30:00-05:00  29.0000  28.495240  28.20  0.8000
2023-02-01 11:25:00-05:00  29.9900  29.460972  29.27  0.7200
2023-02-01 14:40:00-05:00  30.6500  30.064142  29.58  1.0700


In [7]:
# ─── Cell 7: Position Sizing ──────────────────────────────────
import datetime

# ── CONTROL VARIABLES ─────────────────────────────────────────
DOLLAR_RISK  = 100     # 1% of $10K account
MIN_RISK_PCT = 0.002   # 0.2% of price minimum
MAX_RISK_PCT = 0.010   # 1.0% of price maximum

# ── Dynamic risk filters ───────────────────────────────────────
df['min_risk'] = df['close'] * MIN_RISK_PCT
df['max_risk'] = df['close'] * MAX_RISK_PCT

df['valid_signal'] = (
    (df['signal'] == 1) &
    (df['risk'] >= df['min_risk']) &
    (df['risk'] <= df['max_risk'])
).astype(int)

# ── Dynamic sizing ─────────────────────────────────────────────
df['shares'] = (DOLLAR_RISK / df['risk']).clip(lower=1).round(0)
df['shares'] = df['shares'].fillna(1)

valid = df[df['valid_signal']==1]

print(f"✅ Position sizing calculated")
print(f"   Valid signals:  {df['valid_signal'].sum():,}")
print(f"   DOLLAR_RISK:    ${DOLLAR_RISK}")
print(f"   Avg shares:     {valid['shares'].mean():.0f}")
print(f"   Min shares:     {valid['shares'].min():.0f}")
print(f"   Max shares:     {valid['shares'].max():.0f}")
print(f"   Avg stop dist:  ${valid['risk'].mean():.2f}")

print(f"\nRisk distribution:")
signals = valid['risk']
print(f"   < $0.10:          {(signals < 0.10).sum():,} ({(signals < 0.10).mean()*100:.1f}%)")
print(f"   $0.10 - $0.25:    {((signals >= 0.10) & (signals < 0.25)).sum():,} ({((signals >= 0.10) & (signals < 0.25)).mean()*100:.1f}%)")
print(f"   $0.25 - $0.50:    {((signals >= 0.25) & (signals < 0.50)).sum():,} ({((signals >= 0.25) & (signals < 0.50)).mean()*100:.1f}%)")
print(f"   $0.50 - $1.00:    {((signals >= 0.50) & (signals < 1.00)).sum():,} ({((signals >= 0.50) & (signals < 1.00)).mean()*100:.1f}%)")
print(f"   > $1.00:          {(signals >= 1.00).sum():,} ({(signals >= 1.00).mean()*100:.1f}%)")

✅ Position sizing calculated
   Valid signals:  3,478
   DOLLAR_RISK:    $100
   Avg shares:     301
   Min shares:     41
   Max shares:     1667
   Avg stop dist:  $0.44

Risk distribution:
   < $0.10:          17 (0.5%)
   $0.10 - $0.25:    723 (20.8%)
   $0.25 - $0.50:    1,650 (47.4%)
   $0.50 - $1.00:    969 (27.9%)
   > $1.00:          119 (3.4%)


In [8]:
# ─── Cell 8: Scaling Exit Framework + Backtest ────────────────
import numpy as np
import pandas as pd

# ── Scaling Exit Functions ────────────────────────────────────
def is_whole_dollar(price):
    return abs(price - round(price)) < 0.01

def calculate_tp_levels(entry, stop, risk):
    half_risk      = risk / 2
    tp1_target     = entry + half_risk
    nearest_whole  = round(entry)
    if nearest_whole <= entry:
        nearest_whole += 1
    pre_tp1_whole  = nearest_whole if nearest_whole < tp1_target else None
    tp1            = round(tp1_target * 10) / 10
    tps            = []
    current        = tp1
    for _ in range(10):
        tps.append(round(current, 2))
        current = round(current + 0.10, 2)
    return pre_tp1_whole, tps

def simulate_trade(entry, stop, shares, price_series):
    risk             = entry - stop
    pre_tp1, tps     = calculate_tp_levels(entry, stop, risk)
    total_pnl        = 0
    remaining        = shares
    stop_pct_to_be   = 0
    tp_index         = 0
    pre_done         = False
    lot_a = lot_b    = 0
    using_lots       = False

    if pre_tp1 is not None:
        using_lots = True
        lot_a      = shares // 2
        lot_b      = shares - lot_a

    def get_current_stop():
        if stop_pct_to_be == 100:
            return entry
        elif stop_pct_to_be == 50:
            return stop + (risk * 0.5)
        else:
            return stop

    prices = price_series.values if hasattr(price_series, 'values') else price_series

    for price in prices:
        if remaining <= 0:
            break

        active_stop = get_current_stop()

        if price <= active_stop:
            loss       = (active_stop - entry) * remaining
            total_pnl += loss
            remaining  = 0
            break

        if using_lots and not pre_done and pre_tp1 and price >= pre_tp1:
            exit_shares = min(round(shares * 0.25), remaining)
            total_pnl  += (pre_tp1 - entry) * exit_shares
            remaining  -= exit_shares
            lot_a       = remaining // 2
            lot_b       = remaining - lot_a
            stop_pct_to_be = 50
            pre_done    = True
            continue

        if tp_index < len(tps) and price >= tps[tp_index]:
            tp_price = tps[tp_index]
            whole    = is_whole_dollar(tp_price)

            if tp_index == 0:
                exit_pct = 0.60 if whole else 0.50
                if whole:
                    stop_pct_to_be = 100
            elif tp_index == 1:
                exit_pct = 0.30 if whole else 0.20
                stop_pct_to_be = 100 if whole else 50
            elif tp_index == 2:
                exit_pct       = 0.30 if whole else 0.20
                stop_pct_to_be = 100
                if whole:
                    total_pnl += (tp_price - entry) * remaining
                    remaining  = 0
                    break
            elif tp_index == 3:
                exit_pct = 0.20 if whole else 0.10
                if whole:
                    total_pnl += (tp_price - entry) * remaining
                    remaining  = 0
                    break
            else:
                exit_pct = 0.10
                if whole:
                    stop_pct_to_be = 100
                    exit_pct       = min(exit_pct + 0.10, 1.0)

            exit_shares = min(round(remaining * exit_pct), remaining)

            if using_lots and lot_a > 0:
                from_a  = min(exit_shares, lot_a)
                from_b  = exit_shares - from_a
                lot_a  -= from_a
                lot_b  -= max(0, from_b)

            total_pnl += (tp_price - entry) * exit_shares
            remaining -= exit_shares
            tp_index  += 1

            if remaining > 0 and whole and tp_index > 3:
                total_pnl += (tp_price - entry) * remaining
                remaining  = 0
                break

    if remaining > 0:
        last_price = prices[-1]
        exit_price = max(last_price, get_current_stop())
        total_pnl += (exit_price - entry) * remaining

    return total_pnl

# ── Run Backtest ──────────────────────────────────────────────
results   = []
sig_bars  = df[df['valid_signal'] == 1].copy()

for idx, row in sig_bars.iterrows():
    entry  = row['close']
    stop   = row['stop']
    risk   = row['risk']
    if risk <= 0:
        continue
    shares  = max(round(DOLLAR_RISK / risk), 1)
    eod     = idx.replace(hour=16, minute=0, second=0)
    forward = df.loc[idx:eod, 'close'].iloc[1:]
    if len(forward) < 2:
        continue
    pnl = simulate_trade(entry, stop, shares, forward)
    results.append({
        'entry_time':  idx,
        'entry_price': entry,
        'stop':        stop,
        'risk':        risk,
        'shares':      shares,
        'pnl':         pnl
    })

# ── Results ───────────────────────────────────────────────────
results_df          = pd.DataFrame(results)
winners             = results_df[results_df['pnl'] > 0]
losers              = results_df[results_df['pnl'] < 0]
total               = len(results_df)
results_df['cum_pnl'] = results_df['pnl'].cumsum()
max_dd              = (results_df['cum_pnl'] -
                       results_df['cum_pnl'].cummax()).min()

print(f"\n{'='*52}")
print(f"  9-EMA CONTINUATION — NVDL NO FILTER")
print(f"  EMA Slope Only | Full Session | $100 Risk")
print(f"{'='*52}")
print(f"  Total trades:    {total}")
print(f"  Winners:         {len(winners)} ({len(winners)/total*100:.1f}%)")
print(f"  Losers:          {len(losers)} ({len(losers)/total*100:.1f}%)")
print(f"  Avg win:         ${winners['pnl'].mean():.2f}")
print(f"  Avg loss:        ${losers['pnl'].mean():.2f}")
print(f"  Total P&L:       ${results_df['pnl'].sum():.2f}")
print(f"  Max win:         ${results_df['pnl'].max():.2f}")
print(f"  Max loss:        ${results_df['pnl'].min():.2f}")
print(f"  Max drawdown:    ${max_dd:.2f}")
print(f"  Start capital:   $10,000")
print(f"  End capital:     ${10000 + results_df['pnl'].sum():.2f}")
print(f"  Return:          {results_df['pnl'].sum()/10000*100:.1f}%")


  9-EMA CONTINUATION — NVDL NO FILTER
  EMA Slope Only | Full Session | $100 Risk
  Total trades:    3374
  Winners:         1676 (49.7%)
  Losers:          1696 (50.3%)
  Avg win:         $68.73
  Avg loss:        $-84.53
  Total P&L:       $-28164.64
  Max win:         $345.85
  Max loss:        $-100.62
  Max drawdown:    $-29587.05
  Start capital:   $10,000
  End capital:     $-18164.64
  Return:          -281.6%


In [9]:
# ─── Cell 9: In-Sample vs Out-of-Sample Comparison ───────────

in_sample = {
    'period':   '2023-2025 (In-Sample)',
    'trades':   155,
    'win_rate': 65.8,
    'total_pnl': 2803,
    'per_trade': 18.08,
    'return':   28.0
}

out_sample = {
    'period':   '2026 (Out-of-Sample)',
    'trades':   results_df['pnl'].count(),
    'win_rate': round(len(results_df[results_df['pnl']>0]) /
                len(results_df) * 100, 1),
    'total_pnl': round(results_df['pnl'].sum(), 2),
    'per_trade': round(results_df['pnl'].mean(), 2),
    'return':   round(results_df['pnl'].sum() / 10000 * 100, 1)
}

print(f"\n{'='*60}")
print(f"  OUT-OF-SAMPLE VALIDATION RESULTS")
print(f"  Strategy: 9-EMA + Prev High + Pos Open + VWAP + QQQ")
print(f"  Morning Only | $100 Risk/Trade | LOCKED — No Changes")
print(f"{'='*60}")
print(f"\n{'Metric':<25} {'In-Sample':>15} {'Out-of-Sample':>15}")
print(f"{'-'*55}")
print(f"{'Period':<25} {'2023-2025':>15} {'2026':>15}")
print(f"{'Trades':<25} {in_sample['trades']:>15} {out_sample['trades']:>15}")
print(f"{'Win Rate':<25} {in_sample['win_rate']:>14}% {out_sample['win_rate']:>14}%")
print(f"{'Total P&L':<25} ${in_sample['total_pnl']:>14,.2f} ${out_sample['total_pnl']:>14,.2f}")
print(f"{'Per Trade':<25} ${in_sample['per_trade']:>14,.2f} ${out_sample['per_trade']:>14,.2f}")
print(f"{'Return':<25} {in_sample['return']:>14}% {out_sample['return']:>14}%")

# Verdict
print(f"\n{'='*60}")
if out_sample['win_rate'] >= 55 and out_sample['total_pnl'] > 0:
    print(f"  ✅ STRATEGY VALIDATED — Edge holds out-of-sample")
elif out_sample['total_pnl'] > 0:
    print(f"  ⚠️  PARTIAL VALIDATION — Profitable but win rate dropped")
else:
    print(f"  ❌ STRATEGY FAILED — Does not generalize to 2026")
print(f"{'='*60}")


  OUT-OF-SAMPLE VALIDATION RESULTS
  Strategy: 9-EMA + Prev High + Pos Open + VWAP + QQQ
  Morning Only | $100 Risk/Trade | LOCKED — No Changes

Metric                          In-Sample   Out-of-Sample
-------------------------------------------------------
Period                          2023-2025            2026
Trades                                155            3374
Win Rate                            65.8%           49.7%
Total P&L                 $      2,803.00 $    -28,164.64
Per Trade                 $         18.08 $         -8.35
Return                              28.0%         -281.6%

  ❌ STRATEGY FAILED — Does not generalize to 2026


In [13]:
# ─── OFFICIAL IN-SAMPLE BENCHMARK — LOCKED STRATEGY ──────────
import datetime
import numpy as np
import pandas as pd
from datetime import datetime as dt
from alpaca.data.requests import StockBarsRequest
from alpaca.data.timeframe import TimeFrame, TimeFrameUnit

# ── Core EMA ──────────────────────────────────────────────────
df['ema9']             = df['close'].ewm(span=9, adjust=False).mean()
df['ema_slope']        = df['ema9'] - df['ema9'].shift(6)
df['slope_filter']     = df['ema_slope'] > 0.15
df['prev_touched_ema'] = df['low'].shift(1) <= df['ema9'].shift(1)
df['closed_above_ema'] = df['close'] > df['ema9']

# ── VWAP ──────────────────────────────────────────────────────
df['above_vwap'] = df['close'] > df['vwap']

# ── Stop + Risk ───────────────────────────────────────────────
df['stop'] = df['low'].shift(1)
df['risk']  = (df['close'] - df['stop']).clip(lower=0)

# ── Time ──────────────────────────────────────────────────────
df['time'] = df.index.time
df['date'] = df.index.date

# ── Previous Day High ─────────────────────────────────────────
daily_high = df.groupby('date')['high'].max()
df['prev_day_high'] = df['date'].map(
    lambda d: daily_high.get(
        (pd.Timestamp(d) - pd.offsets.BDay(1)).date(),
        np.nan
    )
)
df['above_prev_high'] = df['close'] > df['prev_day_high']

# ── Positive Open ─────────────────────────────────────────────
daily_close = df.groupby('date')['close'].last()
df['prev_day_close'] = df['date'].map(
    lambda d: daily_close.get(
        (pd.Timestamp(d) - pd.offsets.BDay(1)).date(),
        np.nan
    )
)
df['day_open']      = df.groupby('date')['open'].transform('first')
df['positive_open'] = df['day_open'] > df['prev_day_close']

# ── Pull QQQ fresh ────────────────────────────────────────────
print("Pulling QQQ data...")
request_qqq = StockBarsRequest(
    symbol_or_symbols = ["QQQ"],
    timeframe          = TimeFrame(5, TimeFrameUnit.Minute),
    start              = dt(2023, 1, 1),
    end                = dt(2025, 8, 8),
)
bars_qqq   = client.get_stock_bars(request_qqq)
df_qqq_bm  = bars_qqq.df.reset_index()

# Clean QQQ
df_qqq_bm['timestamp'] = pd.to_datetime(df_qqq_bm['timestamp'])
if df_qqq_bm['timestamp'].dt.tz is None:
    df_qqq_bm['timestamp'] = df_qqq_bm['timestamp'].dt.tz_localize('UTC')
df_qqq_bm['timestamp'] = df_qqq_bm['timestamp'].dt.tz_convert('America/New_York')
df_qqq_bm  = df_qqq_bm.set_index('timestamp')
df_qqq_bm  = df_qqq_bm.between_time('09:30', '16:00').dropna()

# QQQ indicators
df_qqq_bm['ema9']       = df_qqq_bm['close'].ewm(span=9, adjust=False).mean()
df_qqq_bm['above_vwap'] = df_qqq_bm['close'] > df_qqq_bm['vwap']
df_qqq_bm['above_ema']  = df_qqq_bm['close'] > df_qqq_bm['ema9']

# Merge into df
df['qqq_above_vwap'] = df_qqq_bm['above_vwap']
df['qqq_above_ema']  = df_qqq_bm['above_ema']
df['qqq_filter']     = (df['qqq_above_vwap'] & df['qqq_above_ema']).fillna(False)
print(f"✅ QQQ merged — {df['qqq_filter'].sum():,} bars with QQQ bullish")

# ── Locked Signal ─────────────────────────────────────────────
morning = (
    (df['time'] >= datetime.time(9, 30)) &
    (df['time'] < datetime.time(12, 0))
)

locked_signal = (
    df['prev_touched_ema']  &
    df['closed_above_ema']  &
    df['slope_filter']      &
    df['above_vwap']        &
    df['above_prev_high']   &
    df['positive_open']     &
    df['qqq_filter']        &
    morning
)

# ── Run Backtest ──────────────────────────────────────────────
results_locked = []

sig_bars = df[
    locked_signal &
    (df['risk'] >= df['close'] * 0.002) &
    (df['risk'] <= df['close'] * 0.010)
].copy()

for idx, row in sig_bars.iterrows():
    entry  = row['close']
    stop   = row['stop']
    risk   = row['risk']
    if risk <= 0:
        continue
    shares  = max(round(100 / risk), 1)
    eod     = idx.replace(hour=16, minute=0, second=0)
    forward = df.loc[idx:eod, 'close'].iloc[1:]
    if len(forward) < 2:
        continue
    pnl = simulate_trade(entry, stop, shares, forward)
    results_locked.append({'entry_time': idx, 'pnl': pnl})

# ── Results ───────────────────────────────────────────────────
r_df    = pd.DataFrame(results_locked)
winners = r_df[r_df['pnl'] > 0]
losers  = r_df[r_df['pnl'] < 0]
total   = len(r_df)
r_df['cum_pnl'] = r_df['pnl'].cumsum()
max_dd  = (r_df['cum_pnl'] - r_df['cum_pnl'].cummax()).min()

print(f"\n{'='*60}")
print(f"  OFFICIAL IN-SAMPLE BENCHMARK — 2023-2025")
print(f"  Strategy: 9-EMA + PH + PO + VWAP + QQQ + Morning")
print(f"  LOCKED — This number does not change")
print(f"{'='*60}")
print(f"  Total signals:  {locked_signal.sum():,}")
print(f"  Valid trades:   {total}")
print(f"  Win rate:       {len(winners)/total*100:.1f}%")
print(f"  Total P&L:      ${r_df['pnl'].sum():.2f}")
print(f"  Per trade:      ${r_df['pnl'].mean():.2f}")
print(f"  Max DD:         ${max_dd:.2f}")
print(f"  Return:         {r_df['pnl'].sum()/10000*100:.1f}%")
print(f"{'='*60}")
print(f"  ⬆️  These are the official numbers for 2026 comparison")

Pulling QQQ data...
✅ QQQ merged — 17,092 bars with QQQ bullish

  OFFICIAL IN-SAMPLE BENCHMARK — 2023-2025
  Strategy: 9-EMA + PH + PO + VWAP + QQQ + Morning
  LOCKED — This number does not change
  Total signals:  299
  Valid trades:   155
  Win rate:       65.8%
  Total P&L:      $2802.67
  Per trade:      $18.08
  Max DD:         $-629.69
  Return:         28.0%
  ⬆️  These are the official numbers for 2026 comparison
